# 01 - EDA and data cleaning

Inspects the UCI **Z-Alizadeh Sani** cohort and documents the data-quality facts the
rest of the pipeline relies on:

- cohort shape and the four target columns,
- class balance per target,
- missingness and duplicates,
- constant / near-constant features that carry no signal,
- the structural relationship between `Cath` and the three vessel labels.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "ml" / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml" / "src"))
DATA = ROOT / "ml" / "data" / "z_alizadeh_sani.xlsx"
MODELS = ROOT / "backend" / "app" / "models"
RESULTS = ROOT / "ml" / "results"
print("repo root:", ROOT)

In [ ]:
df = pd.read_excel(DATA)
df.columns = [str(c).strip() for c in df.columns]
print("shape:", df.shape)
df.head()

## Targets

`Cath` is the overall-CAD label; `LAD`, `LCX` and `RCA` are the per-vessel stenosis
labels. All four are excluded from the feature matrix to prevent target leakage.

In [ ]:
TARGET_COLS = ["LAD", "LCX", "RCA", "Cath"]
for c in TARGET_COLS:
    counts = df[c].value_counts()
    print(f"{c:6s} {dict(counts)}")

## Data quality

The cohort is complete, which is why the pipeline's imputers exist for inference
robustness rather than to repair the training file.

In [ ]:
print("missing cells total   :", int(df.isna().sum().sum()))
print("duplicate rows        :", int(df.duplicated().sum()))
print("duplicate feature rows:", int(df.drop(columns=TARGET_COLS).duplicated().sum()))

features = [c for c in df.columns if c not in TARGET_COLS]
nunique = df[features].nunique().sort_values()
print("\nconstant / near-constant features:")
for col, n in nunique.items():
    if n == 1:
        print(f"  CONSTANT    {col:24s} -> {df[col].unique()[0]!r}")
    elif n == 2 and df[col].value_counts().min() <= 3:
        print(f"  near-const  {col:24s} -> {dict(df[col].value_counts())}")

`Exertional CP` is constant across all patients, so `VarianceThreshold` in the
training pipeline drops it automatically. `CHF` and `LowTH Ang` are retained: they
are near-constant but not degenerate, and tree splits can still use them.

## Structural check: is `Cath` just the OR of the three vessels?

In [ ]:
y = {c: df[c].isin(["Stenotic", "CAD"]).astype(int).to_numpy() for c in TARGET_COLS}
any_vessel = y["LAD"] | y["LCX"] | y["RCA"]
n_agree = int((any_vessel == y["Cath"]).sum())
print(f"Cath == OR(LAD,LCX,RCA) on {n_agree}/{len(df)} rows ({n_agree / len(df):.1%})")
print("Cath=CAD but no vessel stenotic  :", int(((y["Cath"] == 1) & (any_vessel == 0)).sum()))
print("Cath=Normal but a vessel stenotic:", int(((y["Cath"] == 0) & (any_vessel == 1)).sum()))
print("\nvessels involved per patient:")
print(pd.Series(y["LAD"] + y["LCX"] + y["RCA"]).value_counts().sort_index().to_string())

The overall label is almost exactly the disjunction of the three vessel labels. The
dashboard therefore reports **both** a directly-trained CAD head and the
vessel-derived probability `1 - prod(1 - p_v)`, plus whether the two agree, so the
headline number can never silently contradict the 3D view.

## Feature groups

The 55 inputs map onto the modality groups the brief asks for.

In [ ]:
GROUPS = {
    "demographic": ["Age", "Weight", "Length", "Sex", "BMI", "Obesity"],
    "history/risk": ["DM", "HTN", "Current Smoker", "EX-Smoker", "FH", "CRF", "CVA",
                     "Airway disease", "Thyroid Disease", "CHF", "DLP"],
    "examination": ["BP", "PR", "Edema", "Weak Peripheral Pulse", "Lung rales",
                    "Systolic Murmur", "Diastolic Murmur", "Typical Chest Pain",
                    "Dyspnea", "Function Class", "Atypical", "Nonanginal",
                    "Exertional CP", "LowTH Ang"],
    "ecg": ["Q Wave", "St Elevation", "St Depression", "Tinversion", "LVH",
            "Poor R Progression", "BBB"],
    "laboratory": ["FBS", "CR", "TG", "LDL", "HDL", "BUN", "ESR", "HB", "K", "Na",
                   "WBC", "Lymph", "Neut", "PLT"],
    "echocardiographic": ["EF-TTE", "Region RWMA", "VHD"],
}
total = 0
for group, cols in GROUPS.items():
    missing = [c for c in cols if c not in df.columns]
    total += len(cols)
    note = f"  MISSING {missing}" if missing else ""
    print(f"{group:20s} {len(cols):2d} features{note}")
print(f"{'TOTAL':20s} {total:2d} (expected 55)")

## Numeric distributions by CAD status

`EF-TTE` and `Region RWMA` separate the classes most visibly, which is consistent
with the SHAP rankings in notebook 03.

In [ ]:
import matplotlib.pyplot as plt

panel = ["Age", "BP", "EF-TTE", "Region RWMA", "LDL", "HDL", "FBS", "TG"]
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, col in zip(axes.ravel(), panel):
    for label, color, name in [(0, "#16a34a", "Normal"), (1, "#dc2626", "CAD")]:
        ax.hist(df.loc[y["Cath"] == label, col], bins=18, alpha=0.6, color=color, label=name)
    ax.set_title(col, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
axes[0][0].legend(frameon=False, fontsize=8)
fig.suptitle("Feature distributions by angiographic CAD status")
fig.tight_layout()
plt.show()